In [ ]:
# Import libraries

In [ ]:
import sqlite3
import pandas as pd
import json
import time
import ollama

In [ ]:
# ---------- Utility Functions ----------

def execute_query(db_file, query):
    """Executes a SELECT query and returns column names and data."""
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute(query)
    data = cursor.fetchall()
    column_names = [description[0] for description in cursor.description]
    conn.close()
    return column_names, data

def execute_modify_query(db_file, query, params=None):
    """Executes an INSERT/UPDATE/ALTER query on the database."""
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    try:
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        return True
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return False
    finally:
        conn.close()


In [ ]:
# ---------- Step 1: Add Columns If Needed ----------

db_path = "data.db"
new_columns = {
    'Area': 'TEXT',
    'Time': 'TEXT',
    'Relevance': 'TEXT'
}

for column_name, column_type in new_columns.items():
    alter_query = f"ALTER TABLE TBLPapers ADD COLUMN {column_name} {column_type}"
    try:
        execute_modify_query(db_path, alter_query)
        print(f"Column '{column_name}' added.")
    except Exception as e:
        print(f"Skipping column '{column_name}' (might already exist).")

In [ ]:
# ---------- Step 2: Prepare LLM Prompt ----------

def find_possible_entity(title, abstract):
    """Uses Llama 3.1 70B to extract study area, timeframe, and hydrology relevance."""
    prompt = f"""
You need to act as a hydrological researcher. The user will give you the title and abstract of an article, and you need to identify the study area and research timeframe involved in this article.
The research timeframe should be in years and the format of either “2025” or “2025–2030”. If research timeframe is not specified, return “Not specified.”
Furthermore, you need to determine whether the article is related to hydrology or the water cycle. If yes, return “True”; if not, return “False.”
You must provide the results in JSON format, containing three elements: Area, Time, and Relevance.
The title of the article is: '{title}'
The abstract is: '{abstract}'
Only reply with a valid JSON.
"""

    response = ollama.chat(model="llama3.1:70b", messages=[{
        "role": "user",
        "content": prompt
    }])

    return response.get("message", {}).get("content", "").strip("`\n ")

def parse_llm_output(response_text):
    """Parses JSON string from the LLM response."""
    try:
        return json.loads(response_text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"Area": "Unknown", "Time": "Unknown", "Relevance": "Unknown"}

In [ ]:
# ---------- Step 3: Process Each Paper ----------

# NOTE:
# This step processes all papers in the database using the Llama 3.1 70B model.
# Depending on the number of papers (e.g., over 315,000) and the computational capacity of the local machine running Ollama,
# this step may take several days or even weeks to complete.
# For machines with limited GPU memory or running on CPU, the time per paper can be significant.
# It is strongly recommended to test the script on a smaller subset first, or parallelize with caution if resources allow.


query = "SELECT IDPaper, TI, AB FROM TBLPapers"
column_names, records = execute_query(db_path, query)

for paper_id, title, abstract in records:
    print(f"\nProcessing IDPaper {paper_id}")
    start = time.time()
    
    response = find_possible_entity(title, abstract)
    result = parse_llm_output(response)
    
    update_query = """
    UPDATE TBLPapers
    SET Area = ?, Time = ?, Relevance = ?
    WHERE IDPaper = ?
    """
    updated = execute_modify_query(db_path, update_query, (result['Area'], result['Time'], result['Relevance'], paper_id))
    
    if updated:
        print(f"Updated IDPaper {paper_id}: {result}")
    else:
        print(f"Failed to update IDPaper {paper_id}")
    
    print(f"Elapsed time: {round(time.time() - start, 2)} seconds")

print("\nAll records processed.")